# Kaggle – Data on the top
Tu profe ha decidido cambiar de aires y, por eso, ha comprado una tienda de portátiles. Sin embargo, su única especialidad es Data Science, por lo que ha decidido crear un modelo de ML para establecer los mejores precios.

¿Podrías ayudar a tu profe a mejorar ese modelo?

## Métrica: RMSE

$$RMSE = \sqrt{\frac{1}{n}\sum_{i=1}^{n}(y_i - \hat{y}_i)^2}$$

Donde $y_i$ es el valor real y $\hat{y}_i$ es el valor predicho. **Cuanto menor, mejor.**

---
# PARTE 1: Entrenamiento del modelo

## 1. Librerías

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.svm import SVR

## 2. Datos

In [2]:
df = pd.read_csv('./data/train.csv', encoding='latin-1')

### 2.1 Exploración de los datos

In [3]:
df.head()

,laptop_ID,Company,Product,TypeName,Inches,ScreenResolution,Cpu,Ram,Memory,Gpu,OpSys,Weight,Price_in_euros
0,755,HP,250 G6,Notebook,15.6,Full HD 1920x1080,Intel Core i3 6006U 2GHz,8GB,256GB SSD,Intel HD Graphics 520,Windows 10,1.86kg,539.00
1,618,Dell,Inspiron 7559,Gaming,15.6,Full HD 1920x1080,Intel Core i7 6700HQ 2.6GHz,16GB,1TB HDD,Nvidia GeForce GTX 960<U+039C>,Windows 10,2.59kg,879.01
2,909,HP,ProBook 450,Notebook,15.6,Full HD 1920x1080,Intel Core i7 7500U 2.7GHz,8GB,1TB HDD,Nvidia GeForce 930MX,Windows 10,2.04kg,900.00
3,2,Apple,Macbook Air,Ultrabook,13.3,1440x900,Intel Core i5 1.8GHz,8GB,128GB Flash Storage,Intel HD Graphics 6000,macOS,1.34kg,898.94
4,286,Dell,Inspiron 3567,Notebook,15.6,Full HD 1920x1080,Intel Core i3 6006U 2.0GHz,4GB,1TB HDD,AMD Radeon R5 M430,Linux,2.25kg,428.00


In [4]:
print('Filas:', df.shape[0])
print('Columnas:', df.shape[1])

Filas: 912
Columnas: 13


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 912 entries, 0 to 911
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   laptop_ID         912 non-null    int64  
 1   Company           912 non-null    object 
 2   Product           912 non-null    object 
 3   TypeName          912 non-null    object 
 4   Inches            912 non-null    float64
 5   ScreenResolution  912 non-null    object 
 6   Cpu               912 non-null    object 
 7   Ram               912 non-null    object 
 8   Memory            912 non-null    object 
 9   Gpu               912 non-null    object 
 10  OpSys             912 non-null    object 
 11  Weight            912 non-null    object 
 12  Price_in_euros    912 non-null    float64
dtypes: float64(2), int64(1), object(10)
memory usage: 92.8+ KB


In [6]:
df['Price_in_euros'].describe()

count     912.000000
mean     1111.724090
std       687.959172
min       174.000000
25%       589.000000
50%       978.000000
75%      1483.942500
max      6099.000000
Name: Price_in_euros, dtype: float64

In [7]:
df.groupby('Company')['Price_in_euros'].mean().sort_values(ascending=False)

Company
Razer        2987.333333
LG           2299.000000
Google       1879.000000
Microsoft    1716.970000
MSI          1713.425135
Apple        1540.322353
Huawei       1424.000000
Samsung      1423.000000
Toshiba      1276.558824
Xiaomi       1199.616667
Dell         1173.998680
HP           1093.484639
Asus         1068.684050
Lenovo       1053.578416
Fujitsu       769.000000
Acer          607.613514
Mediacom      294.333333
Chuwi         246.945000
Vero          235.400000
Name: Price_in_euros, dtype: float64

### 2.2 Definir X e y

Extraemos más features de las columnas de texto para dar más información al modelo:
- De `Ram` y `Weight` sacamos el número
- De `Cpu` sacamos si es i7/i5/i3 y los GHz
- De `ScreenResolution` sacamos los píxeles totales y si es táctil o IPS
- De `Gpu` sacamos si es Nvidia o Intel
- De `Memory` sacamos si tiene SSD o HDD

In [8]:
# RAM y peso numéricos
df['Ram_GB']    = df['Ram'].str.replace('GB', '').astype(int)
df['Weight_kg'] = df['Weight'].str.replace('kg', '').astype(float)

# CPU: tipo de procesador y velocidad en GHz
df['is_i7']   = df['Cpu'].str.contains('i7', case=False, na=False).astype(int)
df['is_i5']   = df['Cpu'].str.contains('i5', case=False, na=False).astype(int)
df['is_i3']   = df['Cpu'].str.contains('i3', case=False, na=False).astype(int)
cpu_ghz = df['Cpu'].str.extract(r'(\d+\.\d+)GHz')
df['Cpu_GHz'] = pd.to_numeric(cpu_ghz[0], errors='coerce').fillna(2.0)

# Pantalla: píxeles totales, si es táctil, si es IPS
res = df['ScreenResolution'].str.extract(r'(\d{3,4})x(\d{3,4})')
df['res_w']   = pd.to_numeric(res[0], errors='coerce').fillna(1366)
df['res_h']   = pd.to_numeric(res[1], errors='coerce').fillna(768)
df['pixels']  = df['res_w'] * df['res_h']
df['is_touch']= df['ScreenResolution'].str.contains('Touch', case=False, na=False).astype(int)
df['is_ips']  = df['ScreenResolution'].str.contains('IPS', case=False, na=False).astype(int)

# GPU: Nvidia suele ser más cara
df['gpu_nvidia'] = df['Gpu'].str.contains('Nvidia|GeForce|GTX|RTX', case=False, na=False).astype(int)
df['gpu_intel']  = df['Gpu'].str.contains('Intel', case=False, na=False).astype(int)

# Memoria: SSD es más cara que HDD
df['has_ssd'] = df['Memory'].str.contains('SSD|Flash|NVMe', case=False, na=False).astype(int)
df['has_hdd'] = df['Memory'].str.contains('HDD', case=False, na=False).astype(int)

print('Features creadas ✅')

Features creadas ✅


In [9]:
# LabelEncoder para columnas categóricas
les = {}
for col in ['Company', 'TypeName', 'OpSys']:
    le = LabelEncoder()
    df[col + '_enc'] = le.fit_transform(df[col])
    les[col] = le

print('Encoding hecho ✅')

Encoding hecho ✅


In [10]:
# Definir features y target
features = ['Ram_GB', 'Weight_kg', 'Inches', 'Company_enc', 'TypeName_enc', 'OpSys_enc',
            'is_i7', 'is_i5', 'is_i3', 'Cpu_GHz', 'pixels', 'is_touch', 'is_ips',
            'gpu_nvidia', 'gpu_intel', 'has_ssd', 'has_hdd']

X = df[features]
y = df['Price_in_euros']

print('Shape de X:', X.shape)
print('Features:', features)

Shape de X: (912, 17)
Features: ['Ram_GB', 'Weight_kg', 'Inches', 'Company_enc', 'TypeName_enc', 'OpSys_enc', 'is_i7', 'is_i5', 'is_i3', 'Cpu_GHz', 'pixels', 'is_touch', 'is_ips', 'gpu_nvidia', 'gpu_intel', 'has_ssd', 'has_hdd']


### 2.3 Dividir en train y test

In [11]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

print('Tamaño X_train:', X_train.shape)
print('Tamaño X_val:', X_val.shape)

Tamaño X_train: (729, 17)
Tamaño X_val: (183, 17)


## 3. Procesado de datos

> 🚨 **Data leakage:** si usas un scaler, haz **`.fit()` SOLO sobre `X_train`** y luego aplica `.transform()` sobre `X_train` y `X_test` por separado.
>
> Recuerda también que **todo lo que hagas aquí deberás replicarlo después en `test.csv`** (sección 6).

In [12]:
# RandomForest no necesita escalado, el procesado ya está hecho arriba.
print('Features utilizadas:', features)
print('Tipos de datos:')
print(X_train.dtypes)

Features utilizadas: ['Ram_GB', 'Weight_kg', 'Inches', 'Company_enc', 'TypeName_enc', 'OpSys_enc', 'is_i7', 'is_i5', 'is_i3', 'Cpu_GHz', 'pixels', 'is_touch', 'is_ips', 'gpu_nvidia', 'gpu_intel', 'has_ssd', 'has_hdd']
Tipos de datos:
Ram_GB            int64
Weight_kg       float64
Inches          float64
Company_enc       int64
TypeName_enc      int64
OpSys_enc         int64
is_i7             int64
is_i5             int64
is_i3             int64
Cpu_GHz         float64
pixels            int64
is_touch          int64
is_ips            int64
gpu_nvidia        int64
gpu_intel         int64
has_ssd           int64
has_hdd           int64
dtype: object


## 4. Modelado

### 4.1 Entrenamiento

In [13]:
# RandomForest con 200 árboles y profundidad 15
modelo = RandomForestRegressor(n_estimators=200, max_depth=15, random_state=42)
modelo.fit(X_train, y_train)

print('Modelo entrenado ✅')

Modelo entrenado ✅


### 4.2 Métricas

Recuerda que en la competición se evalúa con **RMSE**.

In [14]:
y_pred = modelo.predict(X_val)
rmse = root_mean_squared_error(y_val, y_pred)
print(f'RMSE en validación: {rmse:.2f} €')
print(f'Interpretación: el modelo se equivoca de media {rmse:.0f}€ en sus predicciones')

RMSE en validación: 352.12 €
Interpretación: el modelo se equivoca de media 352€ en sus predicciones


### 4.3 Optimización (up to you 🫰🏻)

In [15]:
# Probamos con más árboles
modelo_v2 = RandomForestRegressor(n_estimators=300, max_depth=20, random_state=42)
modelo_v2.fit(X_train, y_train)

y_pred_v2 = modelo_v2.predict(X_val)
rmse_v2 = root_mean_squared_error(y_val, y_pred_v2)
print(f'RMSE modelo v2 (300 árboles, profundidad 20): {rmse_v2:.2f} €')

if rmse_v2 < rmse:
    modelo_final = modelo_v2
    print('Usaremos el modelo v2 (es mejor)')
else:
    modelo_final = modelo
    print('Usaremos el modelo original (es mejor o igual)')

RMSE modelo v2 (300 árboles, profundidad 20): 352.42 €
Usaremos el modelo original (es mejor o igual)


## 5. Reentrenamiento sobre todos los datos de `train.csv`

Una vez afinado el modelo, reentrenamos con **todos** los datos disponibles antes de predecir sobre `test.csv`.

> ¿Por qué? El split anterior era solo para validar localmente. Para la submission final queremos aprovechar el 100% de los datos de entrenamiento.

In [16]:
modelo_final.fit(X, y)
print('Modelo reentrenado con el 100% de los datos de train ✅')

Modelo reentrenado con el 100% de los datos de train ✅


---
# PARTE 2: Predicción y submission

## 6. Carga los datos de `test.csv`

In [17]:
X_pred = pd.read_csv('./data/test.csv', encoding='latin-1')
X_pred.head()

,laptop_ID,Company,Product,TypeName,Inches,ScreenResolution,Cpu,Ram,Memory,Gpu,OpSys,Weight
0,209,Lenovo,Legion Y520-15IKBN,Gaming,15.6,Full HD 1920x1080,Intel Core i7 7700HQ 2.8GHz,16GB,512GB SSD,Nvidia GeForce GTX 1060,No OS,2.4kg
1,1281,Acer,Aspire ES1-531,Notebook,15.6,1366x768,Intel Celeron Dual Core N3060 1.6GHz,4GB,500GB HDD,Intel HD Graphics 400,Linux,2.4kg
2,1168,Lenovo,V110-15ISK (i3-6006U/4GB/1TB/No,Notebook,15.6,1366x768,Intel Core i3 6006U 2.0GHz,4GB,1TB HDD,Intel HD Graphics 520,No OS,1.9kg
3,1231,Dell,Inspiron 7579,2 in 1 Convertible,15.6,IPS Panel Full HD / Touchscreen 1920x1080,Intel Core i5 7200U 2.5GHz,8GB,256GB SSD,Intel HD Graphics 620,Windows 10,2.191kg
4,1020,HP,ProBook 640,Notebook,14.0,Full HD 1920x1080,Intel Core i5 7200U 2.5GHz,4GB,256GB SSD,Intel HD Graphics 620,Windows 10,1.95kg


## 7. Replica el procesado en `test.csv`

> ⚠️ Usa `.transform()`, **nunca `.fit_transform()`** sobre los datos de test.

In [18]:
# Mismas transformaciones que en train
X_pred['Ram_GB']    = X_pred['Ram'].str.replace('GB', '').astype(int)
X_pred['Weight_kg'] = X_pred['Weight'].str.replace('kg', '').astype(float)

X_pred['is_i7']   = X_pred['Cpu'].str.contains('i7', case=False, na=False).astype(int)
X_pred['is_i5']   = X_pred['Cpu'].str.contains('i5', case=False, na=False).astype(int)
X_pred['is_i3']   = X_pred['Cpu'].str.contains('i3', case=False, na=False).astype(int)
cpu_ghz = X_pred['Cpu'].str.extract(r'(\d+\.\d+)GHz')
X_pred['Cpu_GHz'] = pd.to_numeric(cpu_ghz[0], errors='coerce').fillna(2.0)

res = X_pred['ScreenResolution'].str.extract(r'(\d{3,4})x(\d{3,4})')
X_pred['res_w']   = pd.to_numeric(res[0], errors='coerce').fillna(1366)
X_pred['res_h']   = pd.to_numeric(res[1], errors='coerce').fillna(768)
X_pred['pixels']  = X_pred['res_w'] * X_pred['res_h']
X_pred['is_touch']= X_pred['ScreenResolution'].str.contains('Touch', case=False, na=False).astype(int)
X_pred['is_ips']  = X_pred['ScreenResolution'].str.contains('IPS', case=False, na=False).astype(int)

X_pred['gpu_nvidia'] = X_pred['Gpu'].str.contains('Nvidia|GeForce|GTX|RTX', case=False, na=False).astype(int)
X_pred['gpu_intel']  = X_pred['Gpu'].str.contains('Intel', case=False, na=False).astype(int)

X_pred['has_ssd'] = X_pred['Memory'].str.contains('SSD|Flash|NVMe', case=False, na=False).astype(int)
X_pred['has_hdd'] = X_pred['Memory'].str.contains('HDD', case=False, na=False).astype(int)

# LabelEncoder con .transform() (no .fit_transform())
for col in ['Company', 'TypeName', 'OpSys']:
    X_pred[col + '_enc'] = les[col].transform(X_pred[col])

X_pred_final = X_pred[features]

print('Test procesado ✅')
print('Shape:', X_pred_final.shape)

Test procesado ✅
Shape: (391, 17)


## 8. Genera la submission

### 8.1 ¿Qué formato espera Kaggle?

In [19]:
sample = pd.read_csv('./data/sample_submission.csv', encoding='latin-1')
sample.head()

,laptop_ID,Price_in_euros
0,209,1949.1
1,1281,805.0
2,1168,1101.0
3,1231,1293.8
4,1020,1832.6


### 8.2 Crea tu submission

In [20]:
predicciones = modelo_final.predict(X_pred_final)

submission = sample.copy()
submission['Price_in_euros'] = predicciones

print('Predicciones generadas ✅')
print(f'Min: {predicciones.min():.0f}€ | Max: {predicciones.max():.0f}€ | Media: {predicciones.mean():.0f}€')
submission.head(10)

Predicciones generadas ✅
Min: 218€ | Max: 4864€ | Media: 1145€


,laptop_ID,Price_in_euros
0,209,1364.958826
1,1281,296.162588
2,1168,427.419659
3,1231,1013.160426
4,1020,1144.399750
5,379,428.455757
6,553,958.818600
7,172,1119.887215
8,779,1631.689577
9,609,320.156492


### 8.3 Chequeador

In [21]:
def checker(df_to_submit, sample, filename=None):
    if df_to_submit.shape != sample.shape:
        print(' Shape incorrecto.')
        print(f'   Tu submission: {df_to_submit.shape} | Esperado: {sample.shape}')
        return
    if not (df_to_submit.columns == sample.columns).all():
        print(' Nombres de columnas incorrectos.')
        print(f'   Tus columnas:       {list(df_to_submit.columns)}')
        print(f'   Columnas esperadas: {list(sample.columns)}')
        return
    if not (df_to_submit['laptop_ID'] == sample['laptop_ID']).all():
        print(' Los IDs no coinciden.')
        return
    if filename is None:
        from datetime import datetime
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        filename = f'submission_{timestamp}.csv'
    df_to_submit.to_csv(filename, index=False)
    print(f" ¡Todo correcto! Submission guardada como '{filename}'.")

In [22]:
checker(submission, sample)

 ¡Todo correcto! Submission guardada como 'submission_20260630_134002.csv'.
